# 08 — Eval (RunPod / GPU)

Runs the full BabyLM 2026 evaluation pipeline against a trained model repo and pushes all results back to the same HF model repo.

**What this notebook does:**
1. Installs eval dependencies and downloads eval data
2. Runs fast zero-shot eval (`eval_zero_shot_fast.sh`) on every `chck_NM` revision that exists on Hub
3. Runs full zero-shot eval (`eval_zero_shot.sh`) on the final model (main branch)
4. Runs fine-tuning eval (`eval_finetuning.sh`) on the final model
5. Collates predictions into a submission-ready JSON (`collate_preds.sh`)
6. Pushes the full `results/` directory + submission JSON to the HF model repo

**Can be run on the same pod as notebook 07 or a fresh one.**
All results are pushed to Hub so nothing is lost if the pod is interrupted.

**Compare runs:** use notebook 09 (local) to pull results from multiple repos and plot side-by-side.

In [ ]:
# ── Cell 1: RunPod setup (run once per session) ──
# Uncomment if starting a fresh pod without the eval repo already cloned.

# !git clone https://github.com/babylm-org/babylm-eval.git /workspace/babylm-eval

print('Setup complete.')

In [ ]:
# ── Cell 2: Config ──
import os

HUB_MODEL_REPO = None          # "flakoash/babylm-gpt2-small-run001" — the repo to evaluate
HF_TOKEN       = os.environ.get("HF_TOKEN")
HF_USER        = "flakoash"

BACKEND = "causal"             # GPT-2 is a causal LM
TRACK   = "strict-small"

EVAL_REPO_DIR  = "/workspace/babylm-eval/strict"   # path to the cloned eval repo's strict/ dir
EVAL_DATA_DIR  = f"{EVAL_REPO_DIR}/evaluation_data"
RESULTS_DIR    = f"{EVAL_REPO_DIR}/results"

# Competition revision names for Strict-Small (must match what training pushed).
# chck_1M … chck_9M then chck_10M … chck_100M
FAST_REVISIONS = (
    [f"chck_{i}M"    for i in range(1, 10)] +
    [f"chck_{i*10}M" for i in range(1, 11)]
)

assert HUB_MODEL_REPO, "Set HUB_MODEL_REPO in this cell."
print(f"Model repo : {HUB_MODEL_REPO}")
print(f"Backend    : {BACKEND}  |  Track: {TRACK}")
print(f"Revisions  : {FAST_REVISIONS}")

In [ ]:
# ── Cell 3: Install eval requirements + download eval data ──
import subprocess, sys
from pathlib import Path

def run(cmd, cwd=None, check=True):
    """Run a shell command, stream output, raise on failure."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result

# Install eval dependencies
run(f"{sys.executable} -m pip install -r requirements.txt -q", cwd=EVAL_REPO_DIR)

# Create empty .env so eval_finetuning.sh doesn't fail on 'source ../.env'
env_path = Path(EVAL_REPO_DIR) / ".env"
if not env_path.exists():
    env_path.write_text("")
    print("Created empty .env")

# Download eval data (skips if already present)
fast_dir = Path(EVAL_DATA_DIR) / "fast_eval"
full_dir = Path(EVAL_DATA_DIR) / "full_eval"
if fast_dir.exists() and full_dir.exists():
    print("Eval data already present — skipping download.")
else:
    print("Downloading eval data from Hub...")
    run(f"{sys.executable} scripts/download_evals.py", cwd=EVAL_REPO_DIR)

print("\nEval environment ready.")

In [ ]:
# ── Cell 4: Discover which revisions exist on Hub ──
# Only run fast eval for revisions that were actually pushed during training.

from huggingface_hub import list_repo_refs

remote_branches = {
    b.name
    for b in list_repo_refs(HUB_MODEL_REPO, repo_type="model", token=HF_TOKEN).branches
}

available = [r for r in FAST_REVISIONS if r in remote_branches]
missing   = [r for r in FAST_REVISIONS if r not in remote_branches]

print(f"Available revisions ({len(available)}): {available}")
if missing:
    print(f"Missing   revisions ({len(missing)}): {missing}")
    print("  → these will be skipped (scored as 0 by collate_preds)")

In [ ]:
# ── Cell 5: Fast zero-shot eval — one revision at a time ──
# Runs eval_zero_shot_fast.sh for each available chck_NM revision.
# Already-completed revisions are skipped (results dir already exists).

from pathlib import Path

fast_eval_dir = f"evaluation_data/fast_eval"
skipped, ran, failed = [], [], []

for revision in available:
    # Check if results already exist for this revision
    result_path = Path(RESULTS_DIR) / HUB_MODEL_REPO / revision / "zero_shot"
    if result_path.exists() and any(result_path.iterdir()):
        print(f"  {revision}: results already present — skipping.")
        skipped.append(revision)
        continue

    print(f"\n{'='*60}")
    print(f"Fast zero-shot eval: {revision}")
    print(f"{'='*60}")
    result = run(
        f"bash scripts/eval_zero_shot_fast.sh {HUB_MODEL_REPO} {revision} {BACKEND} {fast_eval_dir}",
        cwd=EVAL_REPO_DIR,
        check=False,
    )
    if result.returncode == 0:
        ran.append(revision)
    else:
        print(f"  ⚠ {revision} failed (exit {result.returncode}) — continuing.")
        failed.append(revision)

print(f"\nFast eval done — ran: {len(ran)}  skipped: {len(skipped)}  failed: {len(failed)}")
if failed:
    print(f"  Failed: {failed}")

In [ ]:
# ── Cell 6: Full zero-shot eval — final model (main branch) ──

full_eval_dir = "evaluation_data/full_eval"
result_path   = Path(RESULTS_DIR) / HUB_MODEL_REPO / "main" / "zero_shot"

if result_path.exists() and any(result_path.iterdir()):
    print("Full zero-shot results already present — skipping.")
else:
    print("Running full zero-shot eval on final model...")
    run(
        f"bash scripts/eval_zero_shot.sh {HUB_MODEL_REPO} {BACKEND} {full_eval_dir}",
        cwd=EVAL_REPO_DIR,
    )
    print("Full zero-shot eval complete.")

In [ ]:
# ── Cell 7: Fine-tuning eval — final model ──
# Runs GLUE tasks (boolq, multirc, rte, wsc, mrpc, qqp, mnli).
# This is the slowest cell — expect 30–90 min on a GPU.

finetune_result_path = Path(RESULTS_DIR) / HUB_MODEL_REPO / "main" / "finetune"

if finetune_result_path.exists() and any(finetune_result_path.iterdir()):
    print("Fine-tuning results already present — skipping.")
else:
    print("Running fine-tuning eval (GLUE)... this may take a while.")
    run(
        f"bash scripts/eval_finetuning.sh --model_path {HUB_MODEL_REPO}",
        cwd=EVAL_REPO_DIR,
    )
    print("Fine-tuning eval complete.")

In [ ]:
# ── Cell 8: Collate predictions → submission JSON ──

print("Collating predictions...")
run(
    f"bash scripts/collate_preds.sh {HUB_MODEL_REPO} {BACKEND} {TRACK}",
    cwd=EVAL_REPO_DIR,
)

# Find the generated submission file
import json
submission_files = list(Path(RESULTS_DIR).rglob("*submission*.json")) + \
                   list(Path(RESULTS_DIR).rglob("*collated*.json"))
if submission_files:
    print(f"Submission file(s): {[str(f) for f in submission_files]}")
else:
    print("No submission JSON found — check results/ dir manually.")

print("Collation done.")

In [ ]:
# ── Cell 9: Print results summary ──

import json
from pathlib import Path

results_root = Path(RESULTS_DIR) / HUB_MODEL_REPO

def _read_score(task_dir: Path) -> float | None:
    """Best-effort: read accuracy or score from any results JSON in a task dir."""
    for jf in task_dir.rglob("*.json"):
        try:
            data = json.loads(jf.read_text())
            for key in ("accuracy", "score", "acc", "f1"):
                if key in data:
                    return round(float(data[key]), 4)
        except Exception:
            continue
    return None

print(f"Results summary for {HUB_MODEL_REPO}")
print("=" * 60)

# Zero-shot (full)
zs_dir = results_root / "main" / "zero_shot"
if zs_dir.exists():
    print("\nZero-shot (final model):")
    for task_dir in sorted(zs_dir.iterdir()):
        score = _read_score(task_dir)
        print(f"  {task_dir.name:30s}: {score if score is not None else 'n/a'}")

# Fine-tuning (full)
ft_dir = results_root / "main" / "finetune"
if ft_dir.exists():
    print("\nFine-tuning (final model):")
    for task_dir in sorted(ft_dir.iterdir()):
        score = _read_score(task_dir)
        print(f"  {task_dir.name:30s}: {score if score is not None else 'n/a'}")

# Fast zero-shot per revision
print("\nFast zero-shot BLiMP by revision:")
for rev in available:
    rev_dir = results_root / rev / "zero_shot"
    if rev_dir.exists():
        blimp_dirs = [d for d in rev_dir.iterdir() if "blimp" in d.name.lower()]
        scores = [_read_score(d) for d in blimp_dirs]
        scores = [s for s in scores if s is not None]
        avg = round(sum(scores) / len(scores), 4) if scores else None
        print(f"  {rev:12s}: blimp_avg={avg}")
    else:
        print(f"  {rev:12s}: no results")

In [ ]:
# ── Cell 10: Push results to HF model repo ──
# Uploads the full results/ directory so the comparison notebook can pull
# everything without needing to re-run eval.

from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()
results_root = Path(RESULTS_DIR) / HUB_MODEL_REPO

if results_root.exists() and any(results_root.iterdir()):
    print(f"Pushing results/ → {HUB_MODEL_REPO} ...")
    api.upload_folder(
        folder_path=str(results_root),
        repo_id=HUB_MODEL_REPO,
        path_in_repo="results",
        repo_type="model",
        token=HF_TOKEN,
        commit_message="eval results",
    )
    print("Results pushed.")

    # Also push any submission JSON found at the top-level results dir
    for jf in Path(RESULTS_DIR).glob("*.json"):
        api.upload_file(
            path_or_fileobj=str(jf),
            path_in_repo=jf.name,
            repo_id=HUB_MODEL_REPO,
            repo_type="model",
            token=HF_TOKEN,
            commit_message=f"submission: {jf.name}",
        )
        print(f"Pushed submission file: {jf.name}")
else:
    print("No results to push — run eval cells first.")